# Data Cleaning

This notebook loads the raw tables, inspects them for quality issues, applies cleaning steps, and writes the results out as cleaned csv files.

In [ ]:
import pandas as pd

## 1. Load Raw Data

In [ ]:
DATA_PATH = 'data/PART2-26-01-26_reorg'

train_hai = pd.read_csv(DATA_PATH + '/train_hai.tsv', sep='\t')
train_participants = pd.read_csv(DATA_PATH + '/train_participants.tsv', sep='\t')

In [ ]:
name_df = {
    'train_hai': train_hai,
    'train_participants': train_participants,
}

## 2. Inspect Raw Data

In [ ]:
for name, df in name_df.items():
    print(f"\n{'=' * 50}")
    print(f'TABLE: {name}  —  {df.shape[0]} rows, {df.shape[1]} columns')
    print(f"{'=' * 50}")
    print(df.dtypes, '\n')
    print('Missing values (%):')
    print((df.isna().mean() * 100).round(2))
    display(df.head(5))

## 3. Clean HAI Table

Three issues to fix:
1. **Missing `timepoint` (~1.9%) and `value` (~0.02%)**. These rows carry no
   usable measurement, so we drop them.
2. **Non-numeric `value` entries**. Coerce to numeric; anything that cannot
   convert becomes NaN and is dropped.
3. **Placeholder strain entries (`'-'`)**. Not real strains; remove them.

In [ ]:
print(f'Rows before cleaning: {len(train_hai)}')

# Drop rows where timepoint or value is missing
train_hai = train_hai.dropna(subset=['timepoint', 'value'])

# Coerce value to numeric; non-convertible entries become NaN
train_hai['value'] = pd.to_numeric(train_hai['value'], errors='coerce')
train_hai = train_hai.dropna(subset=['value'])

# Remove placeholder strain entries
train_hai = train_hai[train_hai['virus_strain'] != '-']

print(f'Rows after cleaning:  {len(train_hai)}')

## 4. Clean Participants Table

Three issues to fix:
1. **`main_publication_author` missing ~48%** — too sparse to be useful; drop
   the column.
2. **Inconsistent `biological_sex` casing** — normalise to lowercase.
3. **Noisy `race` labels** — collapse `'race: unknown'` and `'race: other'`
   into a single `'Unknown'` category.

In [ ]:
print(f'Rows before cleaning: {len(train_participants)}')

# Drop the sparse publication column
train_participants = train_participants.drop(columns=['main_publication_author'])

# Normalise biological_sex
train_participants['biological_sex'] = (
    train_participants['biological_sex'].str.lower().str.strip()
)

# Standardize race labels
train_participants['race'] = train_participants['race'].replace({
    'race: unknown': 'Unknown',
    'race: other': 'Unknown',
})

print(f'Rows after cleaning:  {len(train_participants)}')

## 5. Export Cleaned Data

In [ ]:
train_hai.to_csv('train_hai_cleaned.csv', index=False)
train_participants.to_csv('train_participants_cleaned.csv', index=False)

print('Wrote train_hai_cleaned.csv         ', train_hai.shape)
print('Wrote train_participants_cleaned.csv ', train_participants.shape)